## Prepare for analysis - get models

**Load modules**

In [8]:
import requests
import reframed
import pandas as pd
import warnings
import cobra

import matplotlib.pyplot as plt
from os import listdir
import json
import numpy as np

In [9]:
import subprocess
import time
import select

In [10]:
import sys

sys.path.append("../functions/")
from minimal_medium import minimal_medium

In [11]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Gut microbiome (Venturelli et al. 2018) 

Models for relevant species were collected from the AGORA2 database ([sbml_files_fixed](https://vmh.life/files/reconstructions/AGORA2/version2.01/))

In [12]:
models = {}

for filename in listdir("../models/Venturelli2018/species_used_noO2/"):
    if filename.endswith(".xml"):
        models[filename.replace(".xml","")] = reframed.load_cbmodel("../models/Venturelli2018/species_used_noO2/"+filename,exchange_detection='^R_EX_(?!biomass__40__e__41__$).*')

In [13]:
media_db = pd.read_csv("../data/Venturelli2018/generated/media_db.tsv",sep="\t")

#### Testing for infeasible cycles

In [15]:
comm_test = reframed.Community('test',models.values())

In [23]:
comm_model = comm_test.merged_model
env_empty = reframed.Environment.empty(comm_model)
env_empty.apply(comm_model,inplace=True)

reframed.FBA(comm_model)

Set parameter Username
Academic license - for non-commercial use only - expires 2026-03-27


Objective: -0.0
Status: Optimal

In [24]:
sol = reframed.FVA(comm_model)

In [53]:
rxn_map = {value:key for key,value in comm_test.reaction_map.items()}

In [81]:
for key,value in sol.items():
    if "EX_" in key and (abs(value[0])>1e-5 or abs(value[1])>1e-5):
        print(key)
        print(value)    

In [82]:
ignore = []
for key,value in sol.items():
    if abs(value[0])>1e-5 or abs(value[1])>1e-5:
        
        substrates = comm_model.reactions[key].get_substrates()
        products = comm_model.reactions[key].get_products()
        mets = list(substrates)+list(products)
        for met in mets:
            
            compartment = comm_model.metabolites[met].compartment
            if compartment=="ext":
                organism = rxn_map[key][0]
                ignore.append((organism,met))

In [75]:
ignore = []
for key,value in sol.items():
    if "DM_atp" in key and (abs(value[0])>1e-5 or abs(value[1])>1e-5):
        print(key)
        print(value)
        

R_DM_atp_c__M_Bacteroides_uniformis_ATCC_8492
[0.0, inf]
R_DM_atp_c__M_Bacteroides_vulgatus_ATCC_8482
[0.0, inf]
R_DM_atp_c__M_Collinsella_aerofaciens_ATCC_25986
[0.0, inf]
R_DM_atp_c__M_Faecalibacterium_prausnitzii_A2_165
[0.0, inf]
R_DM_atp_c__M_Eggerthella_lenta_DSM_2243
[0.0, inf]
R_DM_atp_c__M_Prevotella_copri_CB7_DSM_18205
[0.0, inf]
R_DM_atp_c__M_Clostridium_hiranonis_TO_931_DSM_13275
[0.0, inf]
R_DM_atp_c__M_Eubacterium_rectale_ATCC_33656
[0.0, inf]
R_DM_atp_c__M_Desulfovibrio_piger_ATCC_29098
[0.0, inf]
R_DM_atp_c__M_Bacteroides_ovatus_ATCC_8483
[0.0, inf]
R_DM_atp_c__M_Bacteroides_thetaiotaomicron_VPI_5482
[0.0, inf]
R_DM_atp_c__M_Blautia_hydrogenotrophica_DSM_10507
[0.0, inf]


In [85]:
len(comm_model.get_compartment_metabolites("ext"))

685

In [86]:
mets_ignore = [ig[1] for ig in ignore]
len(mets_ignore)

572

In [90]:
model_test = models["Bacteroides_ovatus_ATCC_8483"]
env_empty_test = reframed.Environment.empty(model_test)

env_empty_test.apply(model_test,inplace=True)

reframed.FVA(model_test,reactions=["R_DM_atp_c_"])

{'R_DM_atp_c_': [0.0, -0.0]}

**Remove metabolites - because they are exchanged without any output**

In [91]:
mets = ["M_nzp__91__e__93__","M_anzp__91__e__93__","M_czp__91__e__93__","M_7a_czp__91__e__93__","M_nchlphncl__91__e__93__","M_chlphncl__91__e__93__","M_cholate__91__e__93__","M_12dhchol__91__e__93__", "M_7dhcdchol__91__e__93__","M_3dhcdchol__91__e__93__"]

In [92]:
models_new = {}
for key,model in models.items():
    for met in mets:
        if met not in model.metabolites:
            continue
        print(f"{met}: {model.metabolites[met]}")
        
        rxns = model.get_metabolite_reactions(met)
        
        model.remove_reactions(rxns)
        
        model.remove_metabolite(met)
        
        print("\n")
    model.update()
    models_new[key]=model

M_nzp__91__e__93__: Nitrazepam, Benzalin, Neozepam


M_anzp__91__e__93__: 7-Amino Nitrozepam


M_czp__91__e__93__: Clonazepam


M_7a_czp__91__e__93__: 7-Aminoclonazepam


M_nchlphncl__91__e__93__: Nitrosochloramphenicol


M_chlphncl__91__e__93__: Chloramphenicol


M_cholate__91__e__93__: Cholate


M_nzp__91__e__93__: Nitrazepam, Benzalin, Neozepam


M_anzp__91__e__93__: 7-Amino Nitrozepam


M_czp__91__e__93__: Clonazepam


M_7a_czp__91__e__93__: 7-Aminoclonazepam


M_nchlphncl__91__e__93__: Nitrosochloramphenicol


M_chlphncl__91__e__93__: Chloramphenicol


M_cholate__91__e__93__: Cholate


M_nzp__91__e__93__: Nitrazepam, Benzalin, Neozepam


M_anzp__91__e__93__: 7-Amino Nitrozepam


M_czp__91__e__93__: Clonazepam


M_7a_czp__91__e__93__: 7-Aminoclonazepam


M_nchlphncl__91__e__93__: Nitrosochloramphenicol


M_chlphncl__91__e__93__: Chloramphenicol


M_cholate__91__e__93__: Cholate


M_12dhchol__91__e__93__: 12-Dehydrocholate


M_7dhcdchol__91__e__93__: 7-Dehydrochenodeoxycholate


M_3

In [95]:
comm_test = reframed.Community('test',models_new.values())
comm_model = comm_test.merged_model

In [96]:
reactions_DM = [rxn for rxn in comm_model.reactions if "DM_atp" in rxn]

In [97]:
env_empty = reframed.Environment.empty(comm_model)
env_empty.apply(comm_model,inplace=True)

reframed.FBA(comm_model)

Objective: -0.0
Status: Optimal

In [104]:
sol = reframed.FVA(comm_model,reactions=reactions_DM)

rxn_map = {value:key for key,value in comm_test.reaction_map.items()}

for key,value in sol.items():
    if "DM_atp" in key and (abs(value[0])>1e-5 or abs(value[1])>1e-5):
        print(key)
        print(value)
        

R_DM_atp_c__M_Bacteroides_uniformis_ATCC_8492
[0.0, inf]
R_DM_atp_c__M_Bacteroides_vulgatus_ATCC_8482
[0.0, inf]
R_DM_atp_c__M_Collinsella_aerofaciens_ATCC_25986
[0.0, inf]
R_DM_atp_c__M_Faecalibacterium_prausnitzii_A2_165
[0.0, inf]
R_DM_atp_c__M_Eggerthella_lenta_DSM_2243
[0.0, inf]
R_DM_atp_c__M_Prevotella_copri_CB7_DSM_18205
[0.0, inf]
R_DM_atp_c__M_Clostridium_hiranonis_TO_931_DSM_13275
[0.0, inf]
R_DM_atp_c__M_Eubacterium_rectale_ATCC_33656
[0.0, inf]
R_DM_atp_c__M_Desulfovibrio_piger_ATCC_29098
[0.0, inf]
R_DM_atp_c__M_Bacteroides_ovatus_ATCC_8483
[0.0, inf]
R_DM_atp_c__M_Bacteroides_thetaiotaomicron_VPI_5482
[0.0, inf]
R_DM_atp_c__M_Blautia_hydrogenotrophica_DSM_10507
[0.0, inf]


In [105]:
len(comm_model.get_compartment_metabolites("ext"))

675

## Prepare for analysis - Make media

### Gut microbiome models

1. Test growth on LB medium
2. Define a minimal media where all models can grow

#### Defining a media where all members can grow (NB: Might change slightly for each run)

*Metabolites to add*

**Confirm growth on new medium**

In [9]:
medium_rxns = [rxn.replace("91","40").replace("93","41") for rxn in media_db.compound.values]

env_lb_extend = reframed.Environment.from_reactions(medium_rxns)

for model_id,model in models_new.items():
    env_lb_extend.apply(model,inplace=True,exclusive=True,warning=False)
    sol = reframed.pFBA(model)
    print(f"{model_id}: {sol.values['R_EX_biomass__40__e__41__']}")


Set parameter Username
Academic license - for non-commercial use only - expires 2026-03-27
Bacteroides_uniformis_ATCC_8492: 2.7077667715773406
Bacteroides_vulgatus_ATCC_8482: 1.8110910556272766
Collinsella_aerofaciens_ATCC_25986: 2.1195647297378257
Faecalibacterium_prausnitzii_A2_165: 2.3709472496854405
Eggerthella_lenta_DSM_2243: 0.39514902895412324
Prevotella_copri_CB7_DSM_18205: 2.364717163550213
Clostridium_hiranonis_TO_931_DSM_13275: 0.7601605197088019
Eubacterium_rectale_ATCC_33656: 3.597816854859658
Desulfovibrio_piger_ATCC_29098: 0.4895879066926546
Bacteroides_ovatus_ATCC_8483: 3.0468014821905003
Bacteroides_thetaiotaomicron_VPI_5482: 2.0860564415879077
Blautia_hydrogenotrophica_DSM_10507: 2.8239206162429364


**Save models**

In [10]:
for key, model in models_new.items():
    print(key)
    reframed.save_cbmodel(model,filename="../models/Venturelli2018/species_used_test/"+key+".xml")

Bacteroides_uniformis_ATCC_8492
Bacteroides_vulgatus_ATCC_8482
Collinsella_aerofaciens_ATCC_25986
Faecalibacterium_prausnitzii_A2_165
Eggerthella_lenta_DSM_2243
Prevotella_copri_CB7_DSM_18205
Clostridium_hiranonis_TO_931_DSM_13275
Eubacterium_rectale_ATCC_33656
Desulfovibrio_piger_ATCC_29098
Bacteroides_ovatus_ATCC_8483
Bacteroides_thetaiotaomicron_VPI_5482
Blautia_hydrogenotrophica_DSM_10507


**Use cobrapy to ensure that they are valid**

In [12]:
outputs = {}
for model_file in listdir("../models/Venturelli2018/species_used_test/"):
    if model_file.endswith(".xml"):
        print(model_file)
        output = cobra.io.validate_sbml_model("../models/Venturelli2018/species_used_test/"+model_file)
        outputs[model_file]=output

Bacteroides_uniformis_ATCC_8492.xml
Bacteroides_vulgatus_ATCC_8482.xml
Collinsella_aerofaciens_ATCC_25986.xml
Faecalibacterium_prausnitzii_A2_165.xml
Eggerthella_lenta_DSM_2243.xml
Prevotella_copri_CB7_DSM_18205.xml
Clostridium_hiranonis_TO_931_DSM_13275.xml
Eubacterium_rectale_ATCC_33656.xml
Desulfovibrio_piger_ATCC_29098.xml
Bacteroides_ovatus_ATCC_8483.xml
Bacteroides_thetaiotaomicron_VPI_5482.xml
Blautia_hydrogenotrophica_DSM_10507.xml


### Gut microbiome

In [14]:
with open("../output/Venturelli2018/results_steadiercom/err_file_test.txt",'w') as errfile:
    process = subprocess.Popen("../scripts_bash/steadiercom_parallel_gut_test.sh", 
                               shell=True, 
                               text=True, 
                               executable="/bin/bash", 
                               stdout=subprocess.PIPE, 
                               stderr=errfile,
                               )

    while True:
        line = process.stdout.readline()
        if not line:
            break
        print(line.rstrip(), flush=True)

BH_minus_abundance.tsv
BO_minus_abundance.tsv
loading communities took: 0.0146 seconds
simulating BO- in LB_extend medium
Set parameter Username
Academic license - for non-commercial use only - expires 2026-03-27
loading communities took: 0.0146 seconds
simulating BH- in LB_extend medium
Set parameter Username
Academic license - for non-commercial use only - expires 2026-03-27
Running SteadierCom and gathering data took: 1.11 minutes
Further processing took: 0.00 minutes
Running SteadierCom and gathering data took: 1.22 minutes
Further processing took: 0.00 minutes
BT_minus_abundance.tsv
BU_minus_abundance.tsv
loading communities took: 0.0032 seconds
simulating BU- in LB_extend medium
Set parameter Username
Academic license - for non-commercial use only - expires 2026-03-27
loading communities took: 0.0032 seconds
simulating BT- in LB_extend medium
Set parameter Username
Academic license - for non-commercial use only - expires 2026-03-27
Running SteadierCom and gathering data took: 1.1

In [6]:
with open("../output/Venturelli2018/results_steadiercom/err_file_2_test.txt",'w') as errfile:
    process = subprocess.Popen("../scripts_bash/steadiercom_parallel_gut_find_abundance_test.sh", 
                               shell=True, 
                               text=True, 
                               executable="/bin/bash", 
                               stdout=subprocess.PIPE, 
                               stderr=errfile,
                               )

    while True:
        line = process.stdout.readline()
        if not line:
            break
        print(line.rstrip(), flush=True)

BH_minus_abundance.tsv
BO_minus_abundance.tsv
loading communities took: 0.0027 seconds
simulating BO- in LB_extend medium
Set parameter Username
Academic license - for non-commercial use only - expires 2026-03-27
loading communities took: 0.0027 seconds
simulating BH- in LB_extend medium
Set parameter Username
Academic license - for non-commercial use only - expires 2026-03-27
Running SteadierCom and gathering data took: 167.13 minutes
Further processing took: 0.00 minutes
Running SteadierCom and gathering data took: 179.81 minutes
Further processing took: 0.00 minutes
BU_minus_abundance.tsv
BT_minus_abundance.tsv
loading communities took: 0.0029 seconds
simulating BU- in LB_extend medium
Set parameter Username
Academic license - for non-commercial use only - expires 2026-03-27
loading communities took: 0.0028 seconds
simulating BT- in LB_extend medium
Set parameter Username
Academic license - for non-commercial use only - expires 2026-03-27
Running SteadierCom and gathering data took: